# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maryam271/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*



This is a binary classification task: predict whether a content item is declining.

I use Logistic Regression as the primary model because it is simple, interpretable, and produces probability scores that can be used to rank content items for review.

The model is intentionally kept simple. The goal is to test whether a learned model provides useful signal beyond the Week-4 rule-based baseline, rather than assuming that a more complex model is automatically better.

I exclude `trend_direction`, `trend_pct`, and `is_declining_label` because they define or directly reveal the target. I also exclude `content_id` and `client_id` because they are identifiers, not predictive features.

The main evaluation metric is Precision@50 because the operational use case is prioritization.

In [5]:
# ML-08 — Create the binary target

df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

print("Target created successfully.")

print("\nTarget distribution:")
print(df["is_declining_label"].value_counts())

print("\nTarget distribution (%):")
print(
    df["is_declining_label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nDeclining rows:", df["is_declining_label"].sum())
print("Non-declining rows:", (df["is_declining_label"] == 0).sum())

Target created successfully.

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Target distribution (%):
is_declining_label
1    54.21
0    45.79
Name: proportion, dtype: float64

Declining rows: 16262
Non-declining rows: 13738


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I use a grouped train/test split by `client_id`.

The same client must not appear in both training and testing because content from the same client can share patterns. A random row split could therefore make the model appear stronger than it really is.

Approximately 20% of clients are held out for testing. The test clients are not used to fit the model or calculate preprocessing statistics.

Both the learned model and the Week-4 baseline are evaluated on the same held-out client set using the same metric.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-08 — Grouped train/test split

from sklearn.model_selection import GroupShuffleSplit

TARGET = "is_declining_label"

groups = df["client_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(df, df[TARGET], groups=groups)
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

print("\nTrain clients:", train_df["client_id"].nunique())
print("Test clients:", test_df["client_id"].nunique())

overlap = set(train_df["client_id"]) & set(test_df["client_id"])

print("\nClient overlap:", len(overlap))

assert len(overlap) == 0

print("\nGrouped split validated successfully.")

Train rows: 23837
Test rows: 6163

Train clients: 25
Test clients: 7

Client overlap: 0

Grouped split validated successfully.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I compare Logistic Regression with the Week-4 rule-based baseline on the same held-out client set.

Both methods produce a ranking of content items for review. I use Precision@50 because the practical use case is to inspect a small number of high-priority items first.

The baseline thresholds are calculated from the training data only and then applied to the test clients. The Logistic Regression model is also fitted only on the training clients.

This comparison tests whether the learned model provides useful signal beyond the simpler rule-based approach.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-08 — Train Logistic Regression and compare with Week-4 baseline

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression


# ---------------------------------------------------------
# 1. Define columns
# ---------------------------------------------------------

TARGET = "is_declining_label"

EXCLUDE = {
    TARGET,
    "trend_direction",
    "trend_pct",
    "content_id",
    "client_id"
}

feature_cols = [
    col for col in train_df.columns
    if col not in EXCLUDE
]

X_train = train_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()

y_train = train_df[TARGET]
y_test = test_df[TARGET]

print("Number of candidate features:", len(feature_cols))


# ---------------------------------------------------------
# 2. Convert numeric-looking object columns
# ---------------------------------------------------------

for col in feature_cols:
    if X_train[col].dtype == "object":
        converted = pd.to_numeric(X_train[col], errors="coerce")

        valid_rate = converted.notna().mean()

        if valid_rate >= 0.95:
            X_train[col] = converted
            X_test[col] = pd.to_numeric(X_test[col], errors="coerce")


# ---------------------------------------------------------
# 3. Identify numeric and categorical features
# ---------------------------------------------------------

numeric_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = [
    col for col in X_train.columns
    if col not in numeric_features
]

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))


# ---------------------------------------------------------
# 4. Preprocessing
# ---------------------------------------------------------

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])


# ---------------------------------------------------------
# 5. Logistic Regression
# ---------------------------------------------------------

model = Pipeline([
    ("preprocessor", preprocessor),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        )
    )
])

model.fit(X_train, y_train)

model_scores = model.predict_proba(X_test)[:, 1]


# ---------------------------------------------------------
# 6. Week-4 baseline
# ---------------------------------------------------------

baseline_train = train_df.copy()
baseline_test = test_df.copy()

# Convert CTR to numeric.
baseline_train["ctr_pct"] = pd.to_numeric(
    baseline_train["ctr"],
    errors="coerce"
)

baseline_test["ctr_pct"] = pd.to_numeric(
    baseline_test["ctr"],
    errors="coerce"
)

# Week-4 baseline thresholds from TRAINING data only.
impression_threshold = baseline_train[
    "impressions_90d"
].quantile(0.75)

nonzero_ctr = baseline_train.loc[
    baseline_train["ctr_pct"] > 0,
    "ctr_pct"
]

nonzero_ctr_median = nonzero_ctr.median()

print("\nBaseline thresholds:")
print("Impression threshold:", impression_threshold)
print("Non-zero CTR median:", nonzero_ctr_median)


# Visibility score
max_impressions = baseline_train["impressions_90d"].max()

baseline_test["visibility_score"] = (
    np.log1p(baseline_test["impressions_90d"])
    / np.log1p(max_impressions)
)


# CTR opportunity
baseline_test["ctr_opportunity"] = np.clip(
    (
        nonzero_ctr_median
        - baseline_test["ctr_pct"]
    ) / nonzero_ctr_median,
    0,
    1
)


# Same Week-4 action score
baseline_test["action_score"] = (
    0.5 * baseline_test["visibility_score"]
    + 0.5 * baseline_test["ctr_opportunity"]
)


# ---------------------------------------------------------
# 7. Precision@50 helper
# ---------------------------------------------------------

def precision_at_50(scores, y_true):
    order = np.argsort(-np.asarray(scores))[:50]
    return np.asarray(y_true)[order].mean()


# ---------------------------------------------------------
# 8. Calculate Precision@50
# ---------------------------------------------------------

model_precision_50 = precision_at_50(
    model_scores,
    y_test.to_numpy()
)

baseline_precision_50 = precision_at_50(
    baseline_test["action_score"].to_numpy(),
    y_test.to_numpy()
)


# ---------------------------------------------------------
# 9. Comparison table
# ---------------------------------------------------------

comparison = pd.DataFrame({
    "Method": [
        "Week-4 baseline",
        "Logistic Regression"
    ],
    "Precision@50": [
        baseline_precision_50,
        model_precision_50
    ]
})

print("\nModel comparison:")
display(comparison)

print("\nTop-50 declining items:")
print(
    "Week-4 baseline:",
    int(baseline_precision_50 * 50),
    "of 50"
)

print(
    "Logistic Regression:",
    int(model_precision_50 * 50),
    "of 50"
)

Number of candidate features: 40
Numeric features: 29
Categorical features: 11

Baseline thresholds:
Impression threshold: 3982.0
Non-zero CTR median: 0.25

Model comparison:


,Method,Precision@50
0,Week-4 baseline,0.66
1,Logistic Regression,1.00



Top-50 declining items:
Week-4 baseline: 33 of 50
Logistic Regression: 50 of 50


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

I inspect the model's false positives and false negatives to understand where its ranking is imperfect.

I also inspect the largest Logistic Regression coefficients to identify which available features the model relies on most strongly.

These coefficients describe associations used by the model; they do not prove that a feature causes content to decline.

The model is therefore treated as decision-support rather than as a definitive diagnosis.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-08 — Error analysis and feature interpretation

# ---------------------------------------------------------
# 1. Create prediction table
# ---------------------------------------------------------

error_analysis = test_df[
    [
        "content_id",
        "client_id",
        "trend_direction",
        TARGET
    ]
].copy()

error_analysis["predicted_probability"] = model_scores
error_analysis["predicted_label"] = (
    error_analysis["predicted_probability"] >= 0.5
).astype(int)

# False positives:
# Model predicts declining, but observed label is not declining.
false_positives = error_analysis[
    (error_analysis[TARGET] == 0)
    & (error_analysis["predicted_label"] == 1)
].copy()

# False negatives:
# Model predicts non-declining, but observed label is declining.
false_negatives = error_analysis[
    (error_analysis[TARGET] == 1)
    & (error_analysis["predicted_label"] == 0)
].copy()


print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))


# ---------------------------------------------------------
# 2. Show highest-confidence errors
# ---------------------------------------------------------

print("\nHighest-confidence false positives:")
display(
    false_positives
    .sort_values("predicted_probability", ascending=False)
    .head(10)
)

print("\nHighest-confidence false negatives:")
display(
    false_negatives
    .sort_values("predicted_probability", ascending=True)
    .head(10)
)


# ---------------------------------------------------------
# 3. Inspect Logistic Regression coefficients
# ---------------------------------------------------------

feature_names = model.named_steps[
    "preprocessor"
].get_feature_names_out()

coefficients = model.named_steps[
    "classifier"
].coef_[0]

coef_table = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
})

coef_table["abs_coefficient"] = (
    coef_table["coefficient"].abs()
)

print("\nFeatures with strongest model associations:")
display(
    coef_table
    .sort_values("abs_coefficient", ascending=False)
    .head(20)
)


# ---------------------------------------------------------
# 4. Basic error-rate check
# ---------------------------------------------------------

total_errors = len(false_positives) + len(false_negatives)
error_rate = total_errors / len(test_df)

print("\nError summary:")
print("Total test rows:", len(test_df))
print("Total errors:", total_errors)
print("Error rate:", round(error_rate, 4))

False positives: 800
False negatives: 811

Highest-confidence false positives:


,content_id,client_id,trend_direction,is_declining_label,predicted_probability,predicted_label
3488,content_a0777b0fd936,client_f369cb89fc,stable,0,0.863191,1
11887,content_ce59581533ca,client_8527a891e2,stable,0,0.826267,1
20736,content_41baf0722ad9,client_8527a891e2,stable,0,0.808593,1
12869,content_5d5653c4eb4f,client_4e07408562,stable,0,0.807950,1
7801,content_2dff72da8702,client_4e07408562,stable,0,0.804325,1
18531,content_d10f9ce1e0cd,client_4e07408562,stable,0,0.791719,1
14718,content_b0d9646900d0,client_4e07408562,stable,0,0.791061,1
8139,content_7fa63804b8f1,client_4e07408562,stable,0,0.789519,1
6739,content_f45787e64ac2,client_4e07408562,stable,0,0.787366,1
12332,content_4d9f36001f06,client_8527a891e2,stable,0,0.787209,1



Highest-confidence false negatives:


,content_id,client_id,trend_direction,is_declining_label,predicted_probability,predicted_label
29158,content_e18144cbd19d,client_4e07408562,down,1,0.081111,0
4081,content_917fc1b11fe1,client_e629fa6598,down,1,0.107000,0
3179,content_6fc3e66c8c58,client_f369cb89fc,down,1,0.127056,0
29280,content_ee2303f5bdba,client_8527a891e2,down,1,0.141371,0
24849,content_2f002563e9cd,client_e629fa6598,down,1,0.146038,0
17690,content_c268b1716236,client_e629fa6598,down,1,0.147474,0
23511,content_4de8c62603bf,client_e629fa6598,down,1,0.148945,0
7977,content_961dd6582177,client_f369cb89fc,down,1,0.151694,0
14353,content_076e8dccff24,client_e629fa6598,down,1,0.156732,0
25967,content_dbeaa2257870,client_e629fa6598,down,1,0.157968,0



Features with strongest model associations:


,feature,coefficient,abs_coefficient
15,numeric__impressions_last_30d,-35.873439,35.873439
18,numeric__impressions_prev_30d,29.535083,29.535083
5,numeric__impressions_90d,1.921298,1.921298
70,categorical__position_tier_top_3,-1.253448,1.253448
19,numeric__clicks_prev_30d,1.058300,1.058300
16,numeric__clicks_last_30d,-0.985563,0.985563
37,categorical__main_intent_navigational,-0.547696,0.547696
8,numeric__sessions_90d,0.546320,0.546320
13,numeric__days_with_impressions,0.539104,0.539104
9,numeric__users_90d,-0.538767,0.538767



Error summary:
Total test rows: 6163
Total errors: 1611
Error rate: 0.2614


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.